# Neural Network Month · Project 1
## Notebook 2 of 3 — The Network, From Scratch

Today we build a neural network with nothing but NumPy and look inside it. No framework hides the mechanics. Everything is a matrix multiply, a squashing function, and a rule for assigning blame.

**Architecture:** 3 inputs → 6 hidden neurons → 2 outputs. That is 38 adjustable numbers: 18 weights + 6 biases in the first layer, and 12 weights + 2 biases in the second.

**This notebook:** setup, forward pass, activation functions, backpropagation, and a look at what the hidden neurons learned. Four figures.

## 1 · Setup, shared code, data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# One palette for every figure in the series
NEURO_PALETTE = {
    "input":    "#7A9E7E",  # sage   - input neurons and features
    "hidden":   "#C97B4A",  # clay   - hidden neurons
    "recall":   "#2F6F73",  # teal   - recall target
    "reaction": "#D9A441",  # ochre  - reaction-time target
    "ink":      "#4A3F35",  # bark   - text, seed data, baselines
    "paper":    "#F5EFE6",  # sand   - background
    "grid":     "#D8CFC0",  # stone  - gridlines
}
P = NEURO_PALETTE
NEURO_CMAP = LinearSegmentedColormap.from_list("neuro", [P["recall"], P["paper"], P["hidden"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

In [ ]:
FEATURES = ["sleep_hrs", "study_hrs", "stress"]
TARGETS  = ["recall_score", "reaction_ms"]

def true_process(sleep, study, stress, rng=None):
    """The hidden 'biology' our network must rediscover. Noise only if rng is given."""
    sleep_norm  = np.clip((sleep - 4) / 4, 0, 1)
    sleep_gain  = 25 * (1 - np.exp(-(sleep - 4) / 2))                     # saturating
    study_gain  = 22 * (1 - np.exp(-study / 2)) * (0.5 + 0.5 * sleep_norm) # diminishing + needs sleep
    stress_cost = 0.35 * stress ** 1.5                                     # accelerating
    recall   = 35 + sleep_gain + study_gain - stress_cost
    reaction = 250 + 90 * np.exp(-(sleep - 4) / 2.5) * (1 + 0.1 * (stress - 1)) + 3 * (stress - 1) + 4 * np.maximum(study - 4, 0)
    if rng is not None:
        recall   = recall   + rng.normal(0, 3,  size=np.shape(recall))
        reaction = reaction + rng.normal(0, 8,  size=np.shape(reaction))
    return np.clip(recall, 0, 100), reaction

def make_seed(n, rng):
    """A small 'real-looking' cohort of learners."""
    sleep  = np.clip(rng.normal(6.8, 1.1, n), 4, 9.5)
    study  = rng.uniform(0.5, 6, n)
    stress = np.clip(rng.normal(5, 2, n), 1, 10)
    recall, reaction = true_process(sleep, study, stress, rng)
    return pd.DataFrame({"seed_id": np.arange(n), "sleep_hrs": sleep, "study_hrs": study,
                         "stress": stress, "recall_score": recall, "reaction_ms": reaction})

def smooth_bootstrap(seed_df, n_total, rng, jitter=0.05):
    """Resample whole rows with replacement, then nudge each value by 5% of its spread."""
    cols = FEATURES + TARGETS
    idx = rng.integers(0, len(seed_df), n_total)
    boot = seed_df.iloc[idx].reset_index(drop=True).copy()
    sd = seed_df[cols].std().values
    boot[cols] = boot[cols].values + rng.normal(0, 1, (n_total, len(cols))) * sd * jitter
    boot["sleep_hrs"]    = boot["sleep_hrs"].clip(4, 9.5)
    boot["study_hrs"]    = boot["study_hrs"].clip(0.5, 6)
    boot["stress"]       = boot["stress"].clip(1, 10)
    boot["recall_score"] = boot["recall_score"].clip(0, 100)
    return boot

In [ ]:
def assign_folds(seed_ids, k=5, rng=None):
    """Assign each SEED learner to a fold, so bootstrap copies of one person never straddle train and test."""
    rng = rng or np.random.default_rng(7)
    unique = rng.permutation(np.unique(seed_ids))
    fold_of = {sid: i % k for i, sid in enumerate(unique)}
    return np.array([fold_of[s] for s in seed_ids])

In [ ]:
# Loads the cohort from Notebook 1; rebuilds it identically if the file is missing
if Path("neuro_cohort.csv").exists():
    cohort = pd.read_csv("neuro_cohort.csv")
    seed = pd.read_csv("neuro_seed.csv")
else:
    rng = np.random.default_rng(42)
    seed = make_seed(300, rng)
    cohort = smooth_bootstrap(seed, 1500, rng)
print(f"Seed learners: {len(seed)} | Bootstrapped rows: {len(cohort)}")
cohort.head()

## 2 · The network in one picture

Information flows left to right:

1. Each **input** (sleep, study, stress) is a number.
2. Each **hidden neuron** takes a weighted sum of all three inputs, adds a bias, and passes the result through an activation function.
3. Each **output neuron** takes a weighted sum of the hidden neurons' signals. One output estimates recall, the other reaction time.

Every line in the diagram is a weight the network learns.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.grid(False); ax.axis("off")
ys = {"in": [1.6, 0, -1.6], "hid": np.linspace(2.5, -2.5, 6), "out": [1.0, -1.0]}
xs = {"in": 0, "hid": 1, "out": 2}

for yi in ys["in"]:
    for yh in ys["hid"]:
        ax.plot([xs["in"], xs["hid"]], [yi, yh], color=P["ink"], alpha=0.15, linewidth=1, zorder=1)
for yh in ys["hid"]:
    for yo in ys["out"]:
        ax.plot([xs["hid"], xs["out"]], [yh, yo], color=P["ink"], alpha=0.15, linewidth=1, zorder=1)

ax.scatter([0]*3, ys["in"], s=1400, color=P["input"], edgecolor=P["ink"], zorder=3)
ax.scatter([1]*6, ys["hid"], s=1100, color=P["hidden"], edgecolor=P["ink"], zorder=3)
ax.scatter([2, 2], ys["out"], s=1400, color=[P["recall"], P["reaction"]], edgecolor=P["ink"], zorder=3)

for yi, t in zip(ys["in"], ["Sleep", "Study", "Stress"]):
    ax.text(-0.14, yi, t, ha="right", va="center", fontsize=12)
for yo, t in zip(ys["out"], ["Recall", "Reaction"]):
    ax.text(2.14, yo, t, ha="left", va="center", fontsize=12)
for x, t in [(0, "Input Layer"), (1, "Hidden Layer"), (2, "Output Layer")]:
    ax.text(x, 3.45, t, ha="center", fontsize=12, fontweight="bold")
ax.set_xlim(-0.6, 2.6); ax.set_ylim(-3.2, 3.9)
ax.set_title("Our Tiny Network", fontsize=16, pad=14)
save_fig(fig, "04_network_architecture"); plt.show()

## 3 · Why the squashing function matters

Without an activation function, stacking layers collapses into one big linear equation, so depth would add nothing. A nonlinear activation lets hidden neurons bend the relationship. We use **tanh**, which squashes any input into the range −1 to 1 and, like a real neuron's firing rate, saturates at the extremes.

(That resemblance is loose. An artificial neuron is a very rough abstraction of a biological one. We will be specific about where the analogy holds and where it breaks in the final project of the month.)

In [ ]:
z = np.linspace(-4, 4, 400)
curves = [("Tanh (ours)", np.tanh(z), P["hidden"], 3.0),
          ("ReLU", np.maximum(0, z), P["input"], 2.0),
          ("Sigmoid", 1 / (1 + np.exp(-z)), P["reaction"], 2.0)]

fig, ax = plt.subplots(figsize=(8, 4.6))
for name, y, c, lw in curves:
    ax.plot(z, y, color=c, linewidth=lw, label=name)
ax.axhline(0, color=P["ink"], linewidth=0.8); ax.axvline(0, color=P["ink"], linewidth=0.8)
ax.set_ylim(-1.3, 2.2); ax.set_xlabel("Input to neuron"); ax.set_ylabel("Neuron output")
ax.set_title("Activation Functions Compared"); ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); save_fig(fig, "05_activation_functions"); plt.show()

## 4 · The network class

Read `forward` first, then `backward`.

**Forward** is the prediction. **Backward** is backpropagation: it asks, for every weight, *"if I nudged you up slightly, would the error go up or down, and by how much?"* It answers using the chain rule, working from the output back toward the input. Each weight then takes a small step in the direction that lowers the error. That is gradient descent.

In [ ]:
class TinyNet:
    """3 inputs -> 6 tanh hidden neurons -> 2 linear outputs, trained by backpropagation."""
    def __init__(self, n_in=3, n_hidden=6, n_out=2, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden));      self.b1 = np.zeros(n_hidden)
        self.W2 = r.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, n_out)); self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.X  = X
        self.Z1 = X @ self.W1 + self.b1      # weighted sum into each hidden neuron
        self.A1 = np.tanh(self.Z1)           # hidden neurons "fire"
        self.out = self.A1 @ self.W2 + self.b2
        return self.out

    def backward(self, Y, lr):
        n, n_out = Y.shape
        d_out = 2 * (self.out - Y) / (n * n_out)      # d(loss)/d(output)
        dW2 = self.A1.T @ d_out;  db2 = d_out.sum(0)
        dA1 = d_out @ self.W2.T                       # blame passed back to hidden layer
        dZ1 = dA1 * (1 - self.A1 ** 2)                # tanh derivative
        dW1 = self.X.T @ dZ1;     db1 = dZ1.sum(0)
        self.W1 -= lr * dW1; self.b1 -= lr * db1
        self.W2 -= lr * dW2; self.b2 -= lr * db2

def mse(pred, Y): return float(np.mean((pred - Y) ** 2))

def r2_per_target(pred, Y):
    return 1 - ((Y - pred) ** 2).sum(0) / ((Y - Y.mean(0)) ** 2).sum(0)

def fit_scaler(A): return A.mean(0), A.std(0)

def train(net, Xtr, Ytr, Xva=None, Yva=None, epochs=3000, lr=0.3):
    hist = {"train": [], "val": []}
    for _ in range(epochs):
        pred = net.forward(Xtr)
        hist["train"].append(mse(pred, Ytr))
        if Xva is not None:
            hist["val"].append(mse(net.forward(Xva), Yva))
            net.forward(Xtr)                      # restore cached activations for backprop
        net.backward(Ytr, lr)
    return hist

## 5 · Train it

We hold out one fold of **seed learners** for validation (about 20%), and standardise inputs and outputs using the **training rows only**. Standardising with the full dataset would quietly leak information from the validation set.

In [ ]:
fold = assign_folds(cohort["seed_id"].values, k=5)
tr, va = fold != 0, fold == 0

X = cohort[FEATURES].values; Y = cohort[TARGETS].values
x_mu, x_sd = fit_scaler(X[tr]); y_mu, y_sd = fit_scaler(Y[tr])
Xtr, Xva = (X[tr] - x_mu) / x_sd, (X[va] - x_mu) / x_sd
Ytr, Yva = (Y[tr] - y_mu) / y_sd, (Y[va] - y_mu) / y_sd

net = TinyNet(seed=0)
hist = train(net, Xtr, Ytr, Xva, Yva, epochs=3000, lr=0.3)

pred_va = net.forward(Xva) * y_sd + y_mu
r2 = r2_per_target(pred_va, Y[va])
print(f"Train rows: {tr.sum()} | Validation rows: {va.sum()}")
print(f"Validation R-squared  recall: {r2[0]:.3f}  reaction: {r2[1]:.3f}")

## 6 · Figure 6 — watching it learn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
epochs_axis = np.arange(1, len(hist["train"]) + 1)
ax.plot(epochs_axis, hist["train"], color=P["hidden"], linewidth=2.5, label="Training")
ax.plot(epochs_axis, hist["val"], color=P["ink"], linewidth=2, linestyle="--", label="Validation")
ax.set_xscale("log"); ax.set_ylim(0, 1.2)
ax.set_xlabel("Training epoch (log scale)"); ax.set_ylabel("Error (standardised)")
ax.set_title("Training and Validation Loss"); ax.legend(frameon=False)
fig.tight_layout(); save_fig(fig, "06_loss_curves"); plt.show()

Error drops steeply in the first few hundred epochs, then flattens as the network refines. Validation error sits slightly above training error, which is normal: the network always fits data it has seen a little better than data it hasn't. The warning sign would be validation error *rising* while training error keeps falling. That is **overfitting**: memorising seed learners instead of learning the pattern. We don't see that here.

## 7 · Figure 7 — what the hidden neurons learned

Each hidden neuron has three input weights (how strongly it listens to sleep, study, stress) and two output weights (how strongly it influences recall and reaction). Clay-ish cells push a value up, teal cells push it down. These weights are in standardised units, and individual neurons are rarely cleanly interpretable, so read the *patterns*, not any single cell.

In [ ]:
hid_names = [f"H{i+1}" for i in range(6)]
vmax = max(np.abs(net.W1).max(), np.abs(net.W2).max())

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), gridspec_kw={"height_ratios": [3, 2]}, sharex=True)
for ax, W, rows, title in [(a1, net.W1, ["Sleep", "Study", "Stress"], "Input to Hidden Weights"),
                           (a2, net.W2.T, ["Recall", "Reaction"], "Hidden to Output Weights")]:
    im = ax.imshow(W, cmap=NEURO_CMAP, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows); ax.set_title(title)
    ax.grid(False)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            ax.text(j, i, f"{W[i, j]:.1f}", ha="center", va="center", fontsize=10)
a2.set_xticks(range(6)); a2.set_xticklabels(hid_names)
fig.colorbar(im, ax=[a1, a2], shrink=0.8, label="Weight")
fig.suptitle("What Hidden Neurons Learned", y=1.0)
save_fig(fig, "07_hidden_weights"); plt.show()

**What to notice:** in our run, H2, H4 and H6 weight sleep and stress in opposite directions, and the output weights combine the six neurons in different proportions for recall and for reaction. That is the shared internal representation at work: a handful of intermediate "features" that both outputs draw on. Change the random seed and the individual neurons will look different while performance stays similar, which is exactly why no single neuron is "the sleep neuron".

**A necessary honesty note for a brain-themed month:** backpropagation is a very effective training method, but there is no good evidence the brain does literally this. Neurons do not appear to send precise error signals backward through the same connections they used going forward. We borrow the architecture's inspiration from biology; the learning algorithm is engineering.

**Next:** Notebook 3 stress-tests the network with 5-fold cross-validation and asks it a few "what if" questions.